# EXP1 — Верификация (full + adversarial, день 2)

Выборка делится на две независимые части:

- **Full (83 сценария)** — `sweep.build_cases()`. 15 happy-path + 68 с ровно одним инъецированным нарушением из семи типов SV*. Проверка: детектор находит то, что положил генератор. Regression soundness.
- **Adversarial (19 сценариев)** — `adversarial.build_adversarial_cases()`. Граничные passing-кейсы (threshold=0/100, минимальное date-окно, линейные цепочки, ромб, разные target/type/group у близких политик). Проверка: детектор НЕ бьёт ложную тревогу там, где нарушения нет. False positive здесь — настоящий баг валидатора, не regression.

Combined matrix по 102 сценариям даёт итоговый пикап-показатель, где TN из adversarial усиливают Precision.

In [1]:
import json
import os
import sys
import time
import collections
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
EXPERIMENTS_ROOT = NOTEBOOK_DIR.parent
PROJECT_ROOT = EXPERIMENTS_ROOT.parent
BACKEND_SRC = PROJECT_ROOT / 'code' / 'backend' / 'src'
SCENARIOS_DIR = NOTEBOOK_DIR / 'scenarios'
RESULTS_DIR = NOTEBOOK_DIR / 'results'
PZ_FIGURES_DIR = PROJECT_ROOT / 'pz' / 'figures' / 'exp1'

sys.path.insert(0, str(BACKEND_SRC))
sys.path.insert(0, str(EXPERIMENTS_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))

from owlready2 import World

from services.ontology_core import OntologyCore
from services.cache_manager import CacheManager
from services.reasoning import ReasoningOrchestrator
from services.verification import VerificationService

from _common.generator import generate_scenario
from _common.metrics import (
    aggregate_by_property,
    format_csv,
    format_markdown_table,
    macro_average,
)

import sweep as sweep_mod
import adversarial as adv_mod

SCENARIOS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
PZ_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

full_cases = sweep_mod.build_cases()
adv_cases = adv_mod.build_adversarial_cases()
print(f'Full cases: {len(full_cases)}')
print(f'Adversarial cases: {len(adv_cases)}')

Full cases: 83
Adversarial cases: 19


In [2]:
def run_verify(owl_path: Path, course_id: str, include_subsumption: bool):
    world = World()
    onto = world.get_ontology(f"file://{str(owl_path).replace(os.sep, '/')}").load()
    core = OntologyCore(onto_path=str(owl_path), world=world)
    core.onto = onto
    core.world = world
    reasoner = ReasoningOrchestrator(core.onto)
    service = VerificationService(core, reasoner=reasoner, cache=CacheManager(None))
    t0 = time.perf_counter()
    report = service.verify(course_id, include_subsumption=include_subsumption)
    return report, (time.perf_counter() - t0) * 1000.0

In [3]:
gen_start = time.perf_counter()
full_generated: list = []
for case in full_cases:
    owl_path = SCENARIOS_DIR / f"{case.name}.owl"
    generate_scenario(case.config, owl_path)
    full_generated.append((case, owl_path))
gen_full_duration = time.perf_counter() - gen_start

full_runs: list = []
verify_start = time.perf_counter()
for case, owl_path in full_generated:
    report, duration_ms = run_verify(owl_path, case.config.course_id, case.include_subsumption)
    full_runs.append((case, report, duration_ms))
verify_full_duration = time.perf_counter() - verify_start
print(f'Full: генерация {gen_full_duration:.1f} с, верификация {verify_full_duration:.1f} с, среднее {verify_full_duration*1000.0/len(full_runs):.0f} ms/case')

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib

* Owlready2 * Pellet took 1.1765468120574951 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready2 * Pellet took 1.1437602043151855 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready2 * Pellet took 1.0825927257537842 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready2 * Pellet took 1.0635201930999756 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

...

* Owlready * Adding relation sv5_07_t90.gen_student_advanced satisfies sv5_07_t90.gen_subj_group_sub_group


* Owlready2 * Pellet took 1.0951273441314697 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready * Adding relation sv5_08_t55.gen_student_advanced satisfies sv5_08_t55.gen_subj_group_sub_group


* Owlready * Adding relation sv5_09_t40.gen_student_advanced satisfies sv5_09_t40.gen_subj_group_sub_group
Full: генерация 1.7 с, верификация 93.5 с, среднее 1127 ms/case


* Owlready2 * Pellet took 1.0975544452667236 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [4]:
full_pairs = [(dict(r.properties), c.expected) for c, r, _ in full_runs]
full_matrices = aggregate_by_property(full_pairs)
full_macro = macro_average(full_matrices)
full_md = format_markdown_table(full_matrices)
full_csv = format_csv(full_matrices)
print('### Full (83 scenarios)')
print(full_md)
print()
print(f"macro-avg: precision={full_macro['precision']:.3f} recall={full_macro['recall']:.3f} f1={full_macro['f1']:.3f}")

### Full (83 scenarios)
| Свойство | TP | FP | FN | TN | Precision | Recall | F1 | Support |
|---|---|---|---|---|---|---|---|---|
| consistency | 10 | 0 | 0 | 15 | 1.000 | 1.000 | 1.000 | 10 |
| acyclicity | 10 | 0 | 0 | 15 | 1.000 | 1.000 | 1.000 | 10 |
| reachability | 28 | 0 | 0 | 15 | 1.000 | 1.000 | 1.000 | 28 |
| redundancy | 10 | 0 | 0 | 0 | 1.000 | 1.000 | 1.000 | 10 |
| subsumption | 10 | 0 | 0 | 0 | 1.000 | 1.000 | 1.000 | 10 |
| **macro-avg** | . | . | . | . | 1.000 | 1.000 | 1.000 | . |

macro-avg: precision=1.000 recall=1.000 f1=1.000


In [5]:
def fault_breakdown(runs_list, label_getter) -> str:
    bucket = collections.defaultdict(lambda: {'total': 0, 'correct': 0})
    for case, report, _ in runs_list:
        label = label_getter(case)
        bucket[label]['total'] += 1
        correct = True
        for prop, exp_status in case.expected.items():
            actual = report.properties.get(prop)
            if actual is None or actual.status != exp_status:
                correct = False
                break
        if correct:
            bucket[label]['correct'] += 1
    lines = ['| Класс | Сценариев | Распознано | Accuracy |', '|---|---|---|---|']
    for key in sorted(bucket):
        t = bucket[key]['total']
        c = bucket[key]['correct']
        acc = c / t if t else float('nan')
        lines.append(f'| {key} | {t} | {c} | {acc:.3f} |')
    return '\n'.join(lines)

full_breakdown = fault_breakdown(full_runs, lambda c: c.config.fault or 'happy')
print(full_breakdown)

| Класс | Сценариев | Распознано | Accuracy |
|---|---|---|---|
| happy | 15 | 15 | 1.000 |
| sv1_disjointness | 10 | 10 | 1.000 |
| sv2_cycle | 10 | 10 | 1.000 |
| sv3_atomic_threshold | 10 | 10 | 1.000 |
| sv3_empty_date | 8 | 8 | 1.000 |
| sv3_structural | 10 | 10 | 1.000 |
| sv4_redundant | 10 | 10 | 1.000 |
| sv5_subject | 10 | 10 | 1.000 |


## Adversarial boundary-прогон

19 сценариев, каждый — «почти-нарушение», которое детектор должен распознать как валидное. False positive — настоящая ошибка валидатора, не regression. Выборка делится на группы A/B/C/D: атомарные boundary, near-cycle, non-redundancy, non-subsumption.

In [6]:
gen_adv_start = time.perf_counter()
adv_generated: list = []
for case in adv_cases:
    owl_path = SCENARIOS_DIR / f"{case.name}.owl"
    adv_mod.build_scenario(case, owl_path)
    adv_generated.append((case, owl_path))
gen_adv_duration = time.perf_counter() - gen_adv_start

adv_runs: list = []
verify_adv_start = time.perf_counter()
for case, owl_path in adv_generated:
    report, duration_ms = run_verify(owl_path, case.course_id, case.include_subsumption)
    adv_runs.append((case, report, duration_ms))
verify_adv_duration = time.perf_counter() - verify_adv_start

adv_pairs = [(dict(r.properties), c.expected) for c, r, _ in adv_runs]
adv_matrices = aggregate_by_property(adv_pairs)
adv_macro = macro_average(adv_matrices)
adv_md = format_markdown_table(adv_matrices)
adv_csv = format_csv(adv_matrices)
print('### Adversarial (19 scenarios)')
print(adv_md)
print()
print(f"macro-avg: precision={adv_macro['precision']:.3f} recall={adv_macro['recall']:.3f} f1={adv_macro['f1']:.3f}")

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib

* Owlready2 * Pellet took 1.037074089050293 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\ga

* Owlready2 * Pellet took 1.1915500164031982 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready2 * Pellet took 1.0906846523284912 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready2 * Pellet took 1.0606579780578613 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

...

* Owlready2 * Pellet took 1.1445860862731934 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready2 * Pellet took 1.1145412921905518 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready * Adding relation adv_sub_types_dont_subsume.gen_student_0 satisfies adv_sub_types_dont_subsume.adv_p_sub_date
* Owlready * Adding relation adv_sub_types_dont_subsume.gen_activity_0_0 is_available_for adv_sub_types_dont_subsume.gen_student_0
* Owlready * Adding relation adv_sub_types_dont_subsume.gen_activity_0_0 is_available_for adv_sub_types_dont_subsume.gen_student_1
* Owlready * Adding relation adv_sub_types_dont_subsume.gen_student_1 satisfies adv_sub_types_dont_subsume.adv_p_sub_date


### Adversarial (19 scenarios)
| Свойство | TP | FP | FN | TN | Precision | Recall | F1 | Support |
|---|---|---|---|---|---|---|---|---|
| consistency | 0 | 0 | 0 | 19 | n/a | n/a | n/a | 0 |
| acyclicity | 0 | 0 | 0 | 19 | n/a | n/a | n/a | 0 |
| reachability | 0 | 0 | 0 | 19 | n/a | n/a | n/a | 0 |
| redundancy | 0 | 0 | 0 | 10 | n/a | n/a | n/a | 0 |
| subsumption | 0 | 0 | 0 | 10 | n/a | n/a | n/a | 0 |
| **macro-avg** | . | . | . | . | n/a | n/a | n/a | . |

macro-avg: precision=nan recall=nan f1=nan


* Owlready2 * Pellet took 1.0873279571533203 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [7]:
def adv_group(case):
    name = case.name
    if name.startswith('adv_thr') or name.startswith('adv_date') or name.startswith('adv_aggregate'):
        return 'A_atomic_boundary'
    if name.startswith('adv_chain') or name == 'adv_diamond':
        return 'B_near_cycle'
    if name.startswith('adv_red'):
        return 'C_not_redundant'
    if name.startswith('adv_sub'):
        return 'D_not_subsumed'
    return 'other'

adv_breakdown = fault_breakdown(adv_runs, adv_group)
print(adv_breakdown)

| Класс | Сценариев | Распознано | Accuracy |
|---|---|---|---|
| A_atomic_boundary | 5 | 5 | 1.000 |
| B_near_cycle | 4 | 4 | 1.000 |
| C_not_redundant | 5 | 5 | 1.000 |
| D_not_subsumed | 5 | 5 | 1.000 |


In [8]:
combined_pairs = full_pairs + adv_pairs
combined_matrices = aggregate_by_property(combined_pairs)
combined_macro = macro_average(combined_matrices)
combined_md = format_markdown_table(combined_matrices)
combined_csv = format_csv(combined_matrices)
print('### Combined (full + adversarial, 102 scenarios)')
print(combined_md)
print()
print(f"macro-avg: precision={combined_macro['precision']:.3f} recall={combined_macro['recall']:.3f} f1={combined_macro['f1']:.3f}")

### Combined (full + adversarial, 102 scenarios)
| Свойство | TP | FP | FN | TN | Precision | Recall | F1 | Support |
|---|---|---|---|---|---|---|---|---|
| consistency | 10 | 0 | 0 | 34 | 1.000 | 1.000 | 1.000 | 10 |
| acyclicity | 10 | 0 | 0 | 34 | 1.000 | 1.000 | 1.000 | 10 |
| reachability | 28 | 0 | 0 | 34 | 1.000 | 1.000 | 1.000 | 28 |
| redundancy | 10 | 0 | 0 | 10 | 1.000 | 1.000 | 1.000 | 10 |
| subsumption | 10 | 0 | 0 | 10 | 1.000 | 1.000 | 1.000 | 10 |
| **macro-avg** | . | . | . | . | 1.000 | 1.000 | 1.000 | . |

macro-avg: precision=1.000 recall=1.000 f1=1.000


In [9]:
durations_rows = ['scenario,part,fault_or_group,include_subsumption,duration_ms']
for case, _, duration_ms in full_runs:
    durations_rows.append(
        f"{case.name},full,{case.config.fault or 'happy'},{case.include_subsumption},{duration_ms:.1f}"
    )
for case, _, duration_ms in adv_runs:
    durations_rows.append(
        f"{case.name},adversarial,{adv_group(case)},{case.include_subsumption},{duration_ms:.1f}"
    )
durations_csv = '\n'.join(durations_rows) + '\n'

summary_md = '# EXP1 day 2 — full + adversarial\n\n'
summary_md += '## Full (83 scenarios)\n\n' + full_md + '\n\n'
summary_md += '### Breakdown по типам дефектов\n\n' + full_breakdown + '\n\n'
summary_md += '## Adversarial (19 scenarios)\n\n' + adv_md + '\n\n'
summary_md += '### Breakdown по группам граничных кейсов\n\n' + adv_breakdown + '\n\n'
summary_md += '## Combined (full + adversarial, 102 scenarios)\n\n' + combined_md + '\n\n'
summary_md += (
    f"## Параметры\n\n"
    f"- Full: {len(full_runs)} сценариев, генерация {gen_full_duration:.1f} с, верификация {verify_full_duration:.1f} с, среднее {verify_full_duration*1000.0/len(full_runs):.0f} ms/case\n"
    f"- Adversarial: {len(adv_runs)} сценариев, генерация {gen_adv_duration:.1f} с, верификация {verify_adv_duration:.1f} с, среднее {verify_adv_duration*1000.0/len(adv_runs):.0f} ms/case\n"
)

for target_dir in (RESULTS_DIR, PZ_FIGURES_DIR):
    (target_dir / 'full_day2.csv').write_text(full_csv, encoding='utf-8')
    (target_dir / 'adversarial_day2.csv').write_text(adv_csv, encoding='utf-8')
    (target_dir / 'combined_day2.csv').write_text(combined_csv, encoding='utf-8')
    (target_dir / 'full_day2.md').write_text(summary_md, encoding='utf-8')
    (target_dir / 'full_day2_durations.csv').write_text(durations_csv, encoding='utf-8')

print('Экспортировано:')
print(f'  {RESULTS_DIR}')
print(f'  {PZ_FIGURES_DIR}')

Экспортировано:
  C:\Documents\itmo\vkr\vkr-access-control\experiments\exp1_verification\results
  C:\Documents\itmo\vkr\vkr-access-control\pz\figures\exp1
